In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.insert(0,os.path.abspath('..'))
import torch
import random
import numpy as np
from tqdm import tqdm
from glob import glob
import seaborn as sns
from onnx2torch import convert
import matplotlib.pyplot as plt
from skl2onnx.helpers.onnx_helper import load_onnx_model
from src.service.finetune.finetune_after_stitching import finetune_after_stitching

In [8]:
batch_size = 32
val_batch_size = 64
num_epochs = 3
directory = f'../_results_without_finetune/finetune_{num_epochs}'
os.makedirs(directory, exist_ok=True)

In [4]:
def save_figure(x, y, xlabel, ylabel, title, model_name, filename, directory):
    fig, ax = plt.subplots()
    ax = sns.lineplot(x=x, y=y, ax=ax)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.figure.savefig(f"{directory}/{model_name}/{filename}.png")

In [5]:
random.seed(50)
np.random.seed(24)
torch.manual_seed(77)

In [19]:
# useful_models = []
# for j in glob('../_results_without_finetune/finetune/*/*.txt'):
#     model_name = j.split("/")[3]
#     accuracy = float(j.split("/")[-1].replace(".txt", ""))
#     if accuracy < 0.50:
#         continue
#     useful_models.append(model_name)
# useful_models = list(set(useful_models))

In [ ]:
model_paths = sorted(glob('../_results_without_finetune/1713880127_result_BS_32_MD_16_T_0_TT_0.5_K_5/*.onnx'))
best_accuracy = 0
best_model_name = ""
for index, model_path in tqdm(enumerate(model_paths), position=0, leave=True):
    model_name = model_path.split("/")[-1].split(".")[0]
    # if model_name not in useful_models:
    #     continue
    os.makedirs(f'{directory}/{model_name}', exist_ok=True)
    model = convert(load_onnx_model(model_path))
    train_acc_history, train_loss_history, final_accuracy = finetune_after_stitching(model, batch_size=batch_size, num_epochs=num_epochs, val_batch_size=val_batch_size, feature_extracting=False)
    if final_accuracy > best_accuracy:
        best_accuracy = final_accuracy
        best_model_name = model_name
    with open(f'{directory}/{model_name}/{final_accuracy}.txt', 'w') as f:
        for index, (loss, accuracy) in enumerate(zip(train_loss_history, train_acc_history)):
            f.write(f'Epoch: {index + 1}, Loss: {loss}, Accuracy: {accuracy}\n')
    save_figure(x=range(len(train_acc_history)), y=train_acc_history,
                xlabel="epoch", ylabel="accuracy", title=f"{model_name} accuracy data", 
                model_name=model_name, filename="accuracy", directory=directory)
    save_figure(x=range(len(train_loss_history)), y=train_loss_history,
                xlabel="epoch", ylabel="loss", title=f"{model_name} loss data", 
                model_name=model_name, filename="loss", directory=directory)
    model = model.cpu()
    torch.onnx.export(model, torch.ones(1, 3, 224, 224), f'{directory}/{model_name}/model_ft.onnx')
    del model
    torch.cuda.empty_cache()
with open(f'{directory}/final_result.txt', 'w') as f:
    f.write(f"Best model is {best_model_name} with accuracy {best_accuracy}")

0it [00:00, ?it/s]

Epoch 1/3
----------


100%|████████████████████████████████████████████████████████████████████████████████████████████| 315/315 [00:27<00:00, 11.32it/s]


Loss: 0.5546 Accuracy: 0.7519
Epoch 2/3
----------


100%|████████████████████████████████████████████████████████████████████████████████████████████| 315/315 [00:27<00:00, 11.32it/s]


...

37it [2:18:12, 325.00s/it]

Epoch 1/3
----------


100%|████████████████████████████████████████████████████████████████████████████████████████████| 315/315 [01:54<00:00,  2.75it/s]


Loss: 0.7261 Accuracy: 0.6564
Epoch 2/3
----------


 24%|██████████████████████▋                                                                      | 77/315 [00:27<01:27,  2.73it/s]